# Large Language Models and ChatGPT

**Large language models (LLMs)** are the generative AI systems that power modern text assistants, coding tools, and conversational interfaces. At their core, they are neural networks trained to predict the next token in a sequence — a simple objective that, at sufficient scale, produces remarkably fluent and versatile language behavior.

ChatGPT brought this technology to a broad audience by wrapping a large language model in a conversational interface and refining it to follow instructions, stay on topic, and respond helpfully. Understanding how LLMs work — from pre-training through inference — clarifies both their capabilities and their limitations.

This notebook covers LLMs as autoregressive generative models, the pre-training and scaling principles behind them, the evolution from GPT to ChatGPT, what happens when you submit a prompt, and hands-on demonstrations of next-token prediction and autoregressive generation. Deeper topics such as fine-tuning pipelines, alignment methods, retrieval-augmented generation, and agent frameworks are covered in later notebooks.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.random.seed(7)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. LLMs as Autoregressive Generative Models

A **large language model** is a **generative model** over text. It learns the probability distribution of language and can **sample** new text that resembles what it saw during training.

LLMs are **autoregressive**: each new token is generated conditioned on all tokens that came before it. Using the chain rule of probability, the probability of a full sequence is the product of conditional next-token probabilities:

$$P(t_1, t_2, \ldots, t_n) = \prod_{i=1}^{n} P(t_i \mid t_1, t_2, \ldots, t_{i-1})$$

Generation proceeds one token at a time:

1. Start with a prompt (initial tokens).
2. Compute $P(\text{next token} \mid \text{prompt})$.
3. Sample or select a token from that distribution.
4. Append the token to the context.
5. Repeat until a stop condition (end-of-sequence token, max length, or user stop).

This loop is the same mechanism behind autocomplete, code completion, and chat responses. The model does not retrieve pre-written answers — it **constructs** output token by token from learned statistical patterns.

Because generation is probabilistic, the same prompt can produce different outputs on each run. Temperature and other sampling parameters control how much randomness is injected at each step (covered in detail in the inference notebook).

---

## 2. How Large Language Models Work

### 2.1 Pre-Training

Before an LLM can generate useful text, it undergoes **pre-training** — learning from vast amounts of raw text (books, websites, code, and other corpora). The dominant objective is **causal language modeling**: predict the next token given all previous tokens in the sequence.

During pre-training, the model sees billions or trillions of tokens. It adjusts its internal parameters (weights) to minimize prediction error. No human labels are required — the **next token in the text is the label**. This self-supervised approach is what makes scaling to internet-scale data feasible.

Pre-training teaches the model grammar, facts (as statistical associations), reasoning patterns, coding syntax, and conversational structure — all emergent from the single objective of next-token prediction.

### 2.2 Next-Token Prediction

At each position, the model outputs a vector of **logits** — unnormalized scores over the entire vocabulary (often 50,000–100,000+ tokens). A **softmax** function converts logits into a probability distribution:

$$P(t_i = k \mid t_{<i}) = \frac{e^{z_k}}{\sum_{j} e^{z_j}}$$

where $z_k$ is the logit for token $k$. The token with the highest probability is the model's top prediction, but generation typically **samples** from the full distribution to produce varied, natural-sounding text.

### 2.3 Scale: Parameters, Data, and Compute

The "large" in LLM refers primarily to **parameter count** — the number of learnable weights in the neural network. Modern models range from hundreds of millions to hundreds of billions of parameters.

Three scaling factors interact:

| Factor | Role |
|--------|------|
| **Parameters** | Model capacity — how many patterns the network can represent |
| **Training data** | Diversity and volume of language the model learns from |
| **Compute** | GPU/TPU time required to train and run the model |

Empirical scaling laws show that increasing model size, data, and compute together improves performance predictably — lower perplexity (better prediction), more coherent generation, and broader task capability. This observation drove the race toward ever-larger models in the GPT-3 era and beyond.

The architecture behind most LLMs is the **Transformer** — specifically a **decoder-only** stack of self-attention layers that processes token sequences in parallel during training and one step at a time during generation. The transformer architecture is covered in depth in the AI Foundations Deep Learning notebooks.

---

## 3. What Happens When You Send a Prompt

When you type a message into ChatGPT or call a language model API, a pipeline transforms your text into generated output. Here is the conceptual flow:

```
  Your prompt (text)
        │
        ▼
  ┌─────────────┐
  │  Tokenize   │  Split text into token IDs (subword pieces)
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │   Embed     │  Map each token ID to a dense vector
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │ Transformer │  Stack of self-attention + feed-forward layers
  │   layers    │  Each token attends to all previous tokens
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │   Softmax   │  Convert final-layer logits to probabilities
  └──────┬──────┘
         ▼
  ┌─────────────┐
  │   Sample    │  Pick next token (argmax, temperature, top-p, etc.)
  └──────┬──────┘
         ▼
  Append token → repeat until done → decode tokens back to text
```

**Tokenize.** A tokenizer splits your text into subword tokens and maps each to an integer ID. *"ChatGPT is helpful"* might become `[Chat, G, PT, is, helpful]` — the exact split depends on the tokenizer.

**Embed.** Each token ID is looked up in an **embedding matrix**, producing a vector (typically 768–12,288 dimensions) that represents the token in continuous space.

**Transformer layers.** The embedded sequence passes through many layers of **self-attention** and **feed-forward networks**. Self-attention lets each token position gather information from all previous positions, building contextual representations. Positional information (where each token sits in the sequence) is added via positional encodings or rotary embeddings.

**Softmax.** The final layer produces logits over the vocabulary. Softmax normalizes these into a probability distribution over every possible next token.

**Sample.** A token is chosen from the distribution — greedily (highest probability), with temperature scaling, or via top-k / top-p filtering. The chosen token is appended to the sequence, and the entire process repeats for the next position.

**Decode.** When generation finishes, token IDs are converted back to human-readable text.

This loop runs **autoregressively**: each newly generated token becomes part of the context for predicting the next one. That is why longer outputs take more time and compute — each new token requires a full forward pass through the model.

In [ ]:
# Demo 1: Softmax next-token prediction
# Toy vocabulary and logits — simulating the model's final output layer

vocab = ["the", "cat", "dog", "sat", "ran", "on", "mat", "park"]
logits = np.array([2.1, 0.5, -0.3, 1.8, 0.2, -1.0, 0.9, -0.5])

def softmax(x):
    exp_x = np.exp(x - np.max(x))  # subtract max for numerical stability
    return exp_x / exp_x.sum()

probs = softmax(logits)
top_idx = np.argsort(probs)[::-1]

print("Context: 'the cat' -> predict next token\n")
print(f"{'Token':<8s} {'Logit':>8s} {'Probability':>12s}")
print("-" * 32)
for i in top_idx:
    print(f"{vocab[i]:<8s} {logits[i]:8.2f} {probs[i]:12.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(vocab, logits, color="#aec7e8", edgecolor="white")
axes[0].set_ylabel("Logit (unnormalized score)")
axes[0].set_title("Raw logits from the model's output layer")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(vocab, probs, color="#98df8a", edgecolor="white")
axes[1].set_ylabel("Probability")
axes[1].set_title("After softmax — next-token distribution")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

print(f"\nMost likely next token: '{vocab[top_idx[0]]}' (P = {probs[top_idx[0]]:.4f})")

---

## 4. Autoregressive Generation in Practice

The generation loop is simple in concept but produces complex behavior at scale. A toy model with a small vocabulary and hand-crafted transition probabilities illustrates the mechanism without requiring a multi-billion-parameter network.

At each step, the model consults its learned (or here, predefined) conditional probabilities and samples the next token. The sampled token is appended to the context, shifting the probability distribution for the following step. Repeating this process builds sentences word by word — or character by character in smaller demos.

In [ ]:
# Demo 2: Autoregressive generation loop (word-level toy model)

transition_probs = {
    "<START>": {"the": 0.6, "a": 0.4},
    "the":     {"cat": 0.5, "dog": 0.35, "bird": 0.15},
    "a":       {"cat": 0.45, "dog": 0.40, "bird": 0.15},
    "cat":     {"sat": 0.4, "ran": 0.35, "chased": 0.25},
    "dog":     {"sat": 0.35, "ran": 0.40, "chased": 0.25},
    "bird":    {"sat": 0.3, "flew": 0.5, "chased": 0.2},
    "sat":     {"on": 0.7, "in": 0.3},
    "ran":     {"in": 0.6, "on": 0.4},
    "chased":  {"the": 0.5, "a": 0.5},
    "flew":    {"in": 0.6, "over": 0.4},
    "on":      {"the": 0.55, "a": 0.45},
    "in":      {"the": 0.6, "a": 0.4},
    "over":    {"the": 0.7, "a": 0.3},
}

for ctx in ["on", "in", "over"]:
    transition_probs.setdefault(ctx, {})
    transition_probs[ctx].update({"mat": 0.3, "park": 0.35, "tree": 0.2, "house": 0.15})

TERMINAL = {"mat", "park", "tree", "house"}

def sample_next(context_word, rng):
    """Sample one token from P(next | context) — the core generation step."""
    dist = transition_probs.get(context_word, {})
    if not dist:
        return None
    words = list(dist.keys())
    probs = np.array([dist[w] for w in words])
    probs = probs / probs.sum()
    return rng.choice(words, p=probs)

def generate(start="<START>", max_tokens=8, seed=7):
    rng = np.random.default_rng(seed)
    tokens = []
    current = start
    for step in range(max_tokens):
        next_word = sample_next(current, rng)
        if next_word is None:
            break
        tokens.append(next_word)
        if next_word in TERMINAL:
            break
        current = next_word
    return tokens

print("Autoregressive generation — three samples from the same toy model:\n")
for i in range(3):
    seq = generate(seed=7 + i)
    print(f"  Sample {i + 1}: {' '.join(seq)}")

# Trace one generation step-by-step
print("\nStep-by-step trace (seed=42):")
rng = np.random.default_rng(42)
current = "<START>"
generated = []
for step in range(6):
    dist = transition_probs[current]
    words = list(dist.keys())
    probs = np.array([dist[w] for w in words]) / sum(dist.values())
    next_w = rng.choice(words, p=probs)
    generated.append(next_w)
    print(f"  Step {step + 1}: P(· | '{current}') -> sampled '{next_w}'")
    if next_w in TERMINAL:
        break
    current = next_w
print(f"\n  Final sequence: {' '.join(generated)}")

---

## 5. From GPT to ChatGPT: A Conceptual Evolution

OpenAI's GPT family illustrates how scaling pre-trained transformers and adding alignment stages transformed raw language models into conversational assistants.

| Model | Era (approx.) | Key idea |
|-------|---------------|----------|
| **GPT-1** | 2018 | Demonstrated that pre-training a Transformer decoder on books, then fine-tuning on supervised tasks, transfers well across NLP benchmarks. ~117M parameters. |
| **GPT-2** | 2019 | Showed that scaling parameters (1.5B) and training data produces coherent multi-paragraph text without task-specific fine-tuning. |
| **GPT-3** | 2020 | 175B parameters; **few-shot learning** — the model adapts to new tasks from examples in the prompt alone, without weight updates. |
| **InstructGPT / ChatGPT** | 2022 | Added **instruction tuning** and **RLHF** so the model follows user intent, avoids harmful outputs, and converses naturally. |
| **GPT-4** | 2023 | Larger multimodal model with improved reasoning, longer context, and stronger instruction following. Available via API and ChatGPT Plus. |

The progression follows a clear pattern:

1. **Better architecture** — decoder-only Transformers with efficient attention.
2. **More scale** — parameters, data, and compute grow together.
3. **Better alignment** — post-training stages shape raw predictive models into useful assistants.

Raw GPT-3 could complete text but was difficult to steer. ChatGPT added the layers that make the model **helpful, harmless, and honest** in conversational settings — without changing the underlying autoregressive mechanism.

---

## 6. ChatGPT Specifically

### 6.1 Instruction Tuning

A pre-trained LLM predicts whatever text comes next — which may not match what a user wants. **Instruction tuning** (also called supervised fine-tuning, or SFT) trains the model on examples of *(instruction, ideal response)* pairs.

Examples might include:

- *"Summarize this article in three sentences."* → concise summary
- *"Explain photosynthesis to a 10-year-old."* → simplified explanation
- *"Write a Python function to reverse a string."* → correct code

The model learns to **follow instructions** rather than merely continue text. This stage uses a relatively small curated dataset compared to pre-training, but it dramatically improves usability.

### 6.2 RLHF — Reinforcement Learning from Human Feedback

Instruction tuning alone does not capture all nuances of helpfulness and safety. **RLHF** adds a further refinement stage at a high level:

1. **Collect comparisons.** Human raters rank multiple model outputs for the same prompt (which response is better?).
2. **Train a reward model.** A separate model learns to predict human preferences from these rankings.
3. **Optimize with reinforcement learning.** The language model is fine-tuned to maximize the reward model's score while staying close to the instruction-tuned version (to avoid drifting into nonsense).

RLHF aligns model behavior with human preferences — preferring accurate, helpful, and safe responses over plausible-sounding but unhelpful ones. The full alignment pipeline involves additional techniques covered in the Advanced LLM notebooks.

### 6.3 The Conversational Interface

ChatGPT wraps the aligned model in a **chat interface** with several practical components:

- **System prompt** — hidden instructions that set behavior (tone, safety rules, capabilities).
- **Message history** — prior turns in the conversation are included as context so the model maintains coherence across exchanges.
- **Streaming** — tokens are displayed as they are generated, reducing perceived latency.
- **Moderation** — input and output filters block policy-violating content.

From the user's perspective, ChatGPT feels like a dialogue partner. Under the hood, each message triggers the same autoregressive pipeline: tokenize the full conversation context, run the transformer, sample the next token, repeat.

---

## 7. Context Length and the Attention Window

Every LLM has a maximum **context length** — the number of tokens it can process in a single forward pass. This limit arises from the architecture (attention scales with sequence length) and memory constraints.

Context includes everything the model sees at once: system prompt, conversation history, retrieved documents (if any), and the current user message. Tokens beyond the limit are truncated — the model literally cannot "see" them.

Early GPT models handled ~2,048 tokens. Modern models support 8K, 32K, 128K, or more. Longer context enables analyzing lengthy documents in one pass but increases compute cost quadratically (for standard attention) or linearly (with efficient attention variants).

The dedicated **Tokens and Context Windows** notebook explores tokenization and context management in detail. Here we illustrate how truncation affects what the model can use.

In [ ]:
# Demo 3: Context length illustration — what the model "sees"

def make_conversation(n_turns):
    """Simulate a conversation with n_turns user/assistant exchanges."""
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    for i in range(n_turns):
        messages.append({"role": "user", "content": f"Question {i + 1}: Explain topic {i + 1} briefly."})
        messages.append({"role": "assistant", "content": f"Topic {i + 1} is an important concept in its field. " * 8})
    return messages

def estimate_tokens(text):
    """Rough heuristic: ~4 characters per token for English prose."""
    return max(1, len(text) // 4)

def count_context_tokens(messages):
    return sum(estimate_tokens(m["content"]) + 4 for m in messages)  # +4 for role overhead

context_limits = [2048, 8192, 32768, 128000]
turn_counts = range(1, 31)
token_counts = [count_context_tokens(make_conversation(t)) for t in turn_counts]

plt.figure(figsize=(10, 5))
plt.plot(list(turn_counts), token_counts, "o-", color="#1f77b4", linewidth=2, label="Estimated context tokens")
for limit in context_limits:
    plt.axhline(limit, linestyle="--", alpha=0.7, label=f"{limit:,} token limit")
plt.xlabel("Conversation turns (user + assistant pairs)")
plt.ylabel("Estimated tokens in context")
plt.title("Context grows with conversation length — truncation occurs beyond the limit")
plt.legend(loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

# Show truncation for a 2048-token limit
MAX_CTX = 2048
messages = make_conversation(20)
total = count_context_tokens(messages)
print(f"20-turn conversation ≈ {total:,} tokens (limit: {MAX_CTX:,})")
print(f"Tokens beyond limit: {max(0, total - MAX_CTX):,} — these would be truncated or summarized\n")

running = 0
visible_turns = 0
for msg in reversed(messages):
    t = estimate_tokens(msg["content"]) + 4
    if running + t > MAX_CTX:
        break
    running += t
    if msg["role"] == "user":
        visible_turns += 1

print(f"With a {MAX_CTX:,}-token window, roughly the last {visible_turns} user turns remain fully visible.")
print("Earlier conversation history drops out of context — the model cannot reference it.")

---

## 8. Capabilities and Limitations

Understanding what LLMs can and cannot do sets realistic expectations for using systems like ChatGPT.

**Strengths:**

- Fluent text generation across genres, languages, and formats
- Few-shot adaptation to new tasks via prompt examples
- Code generation, explanation, and debugging assistance
- Summarization, translation, and reformatting

**Limitations:**

- **No real-time knowledge** — training data has a cutoff date; the model does not browse the web unless connected to tools
- **Hallucination** — confident-sounding but incorrect statements
- **Context bounds** — long conversations lose early details when truncated
- **No true understanding** — statistical pattern matching, not grounded reasoning or beliefs
- **Inconsistency** — same prompt can yield different answers across runs

These limitations are not bugs to be patched in a single release — they follow from the architecture and training objective. Later notebooks address mitigation strategies such as grounding, evaluation, and responsible deployment.

---

## 9. Summary

**Large language models** are autoregressive generative models that learn $P(\text{next token} \mid \text{context})$ from massive text corpora during pre-training. The Transformer architecture, scaled to billions of parameters and trillions of training tokens, produces the fluent and versatile behavior that defines modern generative AI.

When you send a prompt, the model **tokenizes** your text, **embeds** each token, processes the sequence through **Transformer layers**, applies **softmax** to obtain next-token probabilities, and **samples** one token at a time — repeating until generation completes.

**ChatGPT** builds on this foundation with **instruction tuning** (learning to follow user requests) and **RLHF** (aligning outputs with human preferences), wrapped in a conversational interface that maintains message history within a finite **context window**.

The evolution from GPT-1 through GPT-4 shows a consistent trajectory: scale pre-training, then align behavior for practical use. The notebooks that follow explore tokens and context windows, model selection, inference parameters, and the advanced topics — fine-tuning, evaluation, grounding, and agents — that turn raw language models into production systems.